In [1]:
import sys
import os
import numpy as np
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))

from gymnasium.vector import SyncVectorEnv
from core.env.core import SnakeEnv
from core.env.types import ObserveType
from agents.q_learning import QLearningAgent

num_envs, total_episodes = 16, 100000
env = SyncVectorEnv(
    [
        lambda i=i: SnakeEnv(
            width=20,
            height=20,
            obs_type=ObserveType.VEC_11,
            num_apples=3,
            num_obstacles=15,
            seed=42 + i,
            reward_options={
                "reward_apple": 5.0,
                "reward_step": -0.01,
                "reward_loop_penalty": -0.1,
                "reward_death_wall": -20.0,
                "reward_death_self": -20.0,
                "reward_shaping_closer": 0.3,
                "reward_shaping_further": -0.1,
                "reward_complete": 100.0,
            },
        )
        for i in range(num_envs)
    ]
)

# Extend exploration phase to 60% of total episodes for better Q-table coverage
epsilon_decay = (0.01 / 1.0) ** (1 / (total_episodes * 0.6))

# A gamma closer to 1 (0.99) helps the agent plan further ahead for apples.
# lr=0.05 is generally more stable for tabular Q-learning over long runs.
agent = QLearningAgent(
    state_dim=2048,
    action_dim=3,
    lr=0.05,
    gamma=0.99,
    epsilon_decay=epsilon_decay,
    seed=42,
)

training_logs, episode_rewards, completed = [], np.zeros(num_envs), 0
obs, infos = env.reset()

best_reward = -np.inf

# Pre-compute powers of 2 for fast batched index calculation
pow2 = 1 << np.arange(11)[::-1]

with tqdm(total=total_episodes, desc="Parallel Training") as pbar:
    while completed < total_episodes:
        # Fast batched state index calculation
        state_indices = obs.dot(pow2)

        # Fast batched epsilon-greedy action selection
        actions = []
        for s_idx in state_indices:
            if agent.rng.random() < agent.epsilon:
                actions.append(int(agent.rng.integers(3)))
            else:
                actions.append(int(np.argmax(agent.q_table[s_idx])))

        next_obs, rewards, terms, truncs, next_infos = env.step(actions)
        next_state_indices = next_obs.dot(pow2)

        # Vectorized-style updates in the loop using pre-calculated state indices
        for i in range(num_envs):
            s_idx = state_indices[i]
            ns_idx = next_state_indices[i]
            a = actions[i]
            r = rewards[i]
            term = terms[i]

            # Manual update bypassing agent.update() string/array overhead
            best_next_action = np.argmax(agent.q_table[ns_idx])
            td_target = r + (0 if term else agent.gamma * agent.q_table[ns_idx][best_next_action])
            td_error = td_target - agent.q_table[s_idx][a]
            agent.q_table[s_idx][a] += agent.lr * td_error

            episode_rewards[i] += r

            if term or truncs[i]:
                completed += 1
                if completed <= total_episodes:
                    pbar.update(1)
                    agent.train()  # Decay epsilon per episode

                    reward_val = episode_rewards[i]
                    training_logs.append(
                        {
                            "episode": completed,
                            "reward": reward_val,
                            "epsilon": agent.epsilon,
                        }
                    )

                    if len(training_logs) >= 100:
                        recent_avg = np.mean([log["reward"] for log in training_logs[-100:]])
                        if recent_avg > best_reward:
                            best_reward = recent_avg
                            agent.save("q_learning_snake_best_2.pkl")

                    if completed % 5000 == 0:
                        recent_avg = (
                            np.mean([log["reward"] for log in training_logs[-100:]])
                            if len(training_logs) >= 100
                            else reward_val
                        )
                        tqdm.write(
                            f"Ep {completed}/{total_episodes} | Avg Reward (last 100): {recent_avg:.2f} | Eps: {agent.epsilon:.3f} | Best Avg: {best_reward:.2f}"
                        )
                episode_rewards[i] = 0

        obs = next_obs

env.close()

Parallel Training:   5%|▌         | 5134/100000 [00:04<01:22, 1143.98it/s]

Ep 5000/100000 | Avg Reward (last 100): -14.78 | Eps: 0.681 | Best Avg: -13.42


Parallel Training:   8%|▊         | 7826/100000 [00:06<01:18, 1172.63it/s]


KeyboardInterrupt: 

In [2]:
agent.save("q_learning_snake.pkl")

In [3]:
from core.utils import save_metrics

save_metrics(training_logs, "q_learning_training_logs.csv")

In [4]:
from core.utils import evaluate_agent

evaluate_agent(agent, seed=67)

Evaluating Agent: 100%|██████████| 100/100 [00:02<00:00, 35.53it/s]


Metric          | Average  | Max      | Std Dev 
------------------------------------------------------------
Rewards         | 226.49   | 487.78   | 104.57  
Apples          | 22.43    | 50.00    | 10.01   
Steps           | 329.99   | 864.00   | 163.66  

Death Distribution:
 - self: 87 (87.0%)
 - wall: 13 (13.0%)



({'avg': 226.49129999999977,
  'max': 487.77999999999497,
  'sd': 104.56795798575108},
 {'avg': 22.43, 'max': 50.0, 'sd': 10.009250721207858},
 {'avg': 329.99, 'max': 864.0, 'sd': 163.66053250554944},
 {<DeathReason.SELF: 'self'>: 87, <DeathReason.WALL: 'wall'>: 13})